In [1]:

import pandas as pd
import numpy as np
import re

from gensim.models import Word2Vec
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load dataset
data = pd.read_csv(r"D:\DL\LP IV Updated Datasets\CBOW\twitter_dataset.csv")
data = data.dropna()

texts = data['Text'].tolist()

# 2. Preprocessing
stop_words = set(ENGLISH_STOP_WORDS)

def preprocess(text):
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return [word for word in words if word not in stop_words]

sentences = [preprocess(text) for text in texts]

# 3. Train CBOW Word2Vec
word2vec = Word2Vec(
    sentences=sentences,
    vector_size=100,
    min_count=1,
    sg=0,
    window=5,
    epochs=10
)

print("Vocabulary size:", len(word2vec.wv))

# 4. Convert document into embedding
def get_embedding(words):
    vectors = [word2vec.wv[word] for word in words
               if word in word2vec.wv]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

x = np.array([get_embedding(words) for words in sentences])

# 5. User input
query = input("Enter keywords: ")
query_emb = get_embedding(preprocess(query))

# 6. Calculate cosine similarity
scores = cosine_similarity([query_emb], x)[0]

# 7. Recommend top 5 tweets
for i in scores.argsort()[::-1][:5]:
    print(round(scores[i], 3), "->", texts[i][:100])


ModuleNotFoundError: No module named 'gensim'